In [ ]:
print("Okay")

In [ ]:
## Import libraries

import os
from dotenv import load_dotenv


In [ ]:
load_dotenv()

In [ ]:
# LLM API KEY
groq_api_key = os.getenv("GROQ_API_KEY")

# EMBEDDING API KEY
jina_api_key = os.getenv("JINA_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set. Add it to your .env file.")

if not jina_api_key:
    raise ValueError("JINA_API_KEY is not set. Add it to your .env file.")

print("API keys loaded successfully.")

In [ ]:
# Load the data

LOAD_FILE_PATH = os.path.join("data", "hr_policy.txt")

In [ ]:
# DATA INGESTION
from langchain_community.document_loaders import TextLoader
text_loader = TextLoader(LOAD_FILE_PATH, encoding="utf-8")

documents = text_loader.load()

In [ ]:
print(documents)

In [ ]:
print(documents[0].metadata)

In [ ]:
print(documents[0].page_content)

In [ ]:
len(documents)

In [ ]:
print(f"Total characters in document: {len(documents[0].page_content)}")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)
chunks = text_splitter.split_documents(documents)


In [ ]:
print(chunks)
print(len(chunks))

In [ ]:
print(chunks[0])

In [ ]:
# Embedding data

from langchain_community.embeddings import JinaEmbeddings

In [ ]:
embeddings_model = JinaEmbeddings(
    model_name="jina-embeddings-v3"
)

print("embeddings model name is", embeddings_model.model_name)

In [ ]:
query = embeddings_model.embed_query("What is model")

In [ ]:
print(query)

In [ ]:
# Store the data 
from langchain_community.vectorstores import FAISS

In [ ]:
vectorstor_faiss = FAISS.from_documents(
    chunks,
    embeddings_model    
)

print("Chunks are stored", vectorstor_faiss.index.ntotal)

In [ ]:
test_query = ("How many sick leaves employees get")

top_matches = vectorstor_faiss.similarity_search(
    test_query,
    top_k = 2
)
print(f"Query: {test_query}\n")

for i, match in enumerate(top_matches, start=1):
    print(f"=====Match {i}======")
    print(match.page_content)
    print()

In [ ]:
## TOOL

retriever = vectorstor_faiss.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str)->str:

    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [ ]:
## DATA RETRIEVAL

from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    model= "openai/gpt-oss-120b",
    temperature=0
)

In [ ]:
llm.model_name

In [ ]:
test_response = llm.invoke("How many sick leaves employees get?")

In [ ]:
test_response.content

In [ ]:
from langchain.agents import create_agent
hr_assistant = create_agent(
    model= llm,
    tools=[search_hr_policy],
    system_prompt="""
    You are a friendly HR assistant
    Always use the search_hr_policy tool to look up
    "facts before answering. If the answer isn't in the search results,
    say you don't know the answer instead of guessing."
    """
)

In [ ]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("=" * 60)

    response = hr_assistant.invoke(
        {"messages": [{
            "role": "user",
            "content": question
        }]}
    )
    answer = response["messages"][-1].content

    print("Answer:", answer)
    print("=" * 60)
    return answer

In [ ]:
answer = ask_hr_assistant("How many sick leaves do employees get?")